In [ ]:
!pip install transformers torch sentence-transformers pandas numpy tqdm accelerate -q


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/gdrive/Shareddrives/FML_FINAL/Data/extra_movies_raw.csv")
print(df.shape)
print(df.head())
print(df.dtypes)

(8592, 27)
    tmdb_id    imdb_id                                      title  \
0   97367.0  tt1817273                 The Place Beyond the Pines   
1  203801.0  tt1638355                    The Man from U.N.C.L.E.   
2  191714.0  tt2350496                               The Lunchbox   
3  524348.0  tt8236336                                 The Report   
4  520900.0  tt6439020  The Personal History of David Copperfield   

   release_year release_date  budget_raw    gross_raw   budget_2025  \
0        2013.0   2013-03-20  15000000.0   47135966.0  2.089286e+07   
1        2015.0   2015-08-13  75000000.0  110045109.0  1.013308e+08   
2        2013.0   2013-09-20   3400000.0   17240000.0  4.735714e+06   
3        2019.0   2019-09-12   8000000.0     275000.0  1.018877e+07   
4        2019.0   2019-11-07  15600000.0   11620337.0  1.986810e+07   

     gross_2025  cpi_gross_year  ...  popularity  vote_average  vote_count  \
0  6.469563e+07           233.0  ...      4.5659         6.983      5

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

MODEL_NAME = "Qwen/Qwen3-Embedding-4B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model... (this takes a few minutes)")
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision — fits in T4's 15GB
    device_map="auto"            # automatically puts model on GPU
)
model.eval()

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded on: cuda:0
GPU memory used: 8.0 GB


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import gc
import os

INSTRUCTION = (
    "Instruct: Represent this screenplay for predicting "
    "box office ROI based on genre, tone, and narrative structure\n"
    "Query: "
)

# L4 has 24GB VRAM — after Qwen3-4B weights (~8.5GB)
# we have ~15GB free, so 8000 token chunks are safe
MAX_CHUNK_TOKENS = 8000
OVERLAP_TOKENS = 200

# Pre-compute instruction token length once
INSTRUCTION_TOKEN_LEN = len(tokenizer.encode(INSTRUCTION))


def read_script(path):
    """
    Read script file from disk.
    Tries the path as-is first, then looks in /content/scripts/
    Returns text string or None if file not found.
    """
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()
    except FileNotFoundError:
        alt_path = f"/content/scripts/{os.path.basename(path)}"
        try:
            with open(alt_path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read().strip()
        except FileNotFoundError:
            print(f"  WARNING: Could not find script at {path}")
            return None


def embed_single_chunk(text):
    """
    Embed one chunk of text.
    Assumes text already fits within MAX_CHUNK_TOKENS.
    Clears GPU memory before returning.
    Returns numpy array of shape (2560,).
    """
    full_input = INSTRUCTION + text

    inputs = tokenizer(
        full_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_CHUNK_TOKENS + INSTRUCTION_TOKEN_LEN,
        padding=False
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Qwen3 uses last token pooling
    embeddings = outputs.last_hidden_state[:, -1, :]
    embeddings = F.normalize(embeddings, p=2, dim=1)

    # Pull off GPU immediately
    result = embeddings.squeeze().cpu().float().numpy()

    # Explicit cleanup
    del inputs, outputs, embeddings
    torch.cuda.empty_cache()
    gc.collect()

    return result


def embed_script(script_text):
    """
    Main embedding function.
    Handles scripts of any length via overlapping chunks.
    Returns averaged numpy vector of shape (2560,) or None if all chunks fail.
    """
    script_tokens = tokenizer.encode(script_text, add_special_tokens=False)
    total_tokens = len(script_tokens)

    # Case 1: fits in one chunk
    if total_tokens <= MAX_CHUNK_TOKENS:
        return embed_single_chunk(script_text)

    # Case 2: needs chunking
    chunks_tokens = []
    start = 0

    while start < total_tokens:
        end = min(start + MAX_CHUNK_TOKENS, total_tokens)
        chunks_tokens.append(script_tokens[start:end])
        if end == total_tokens:
            break
        start = end - OVERLAP_TOKENS

    print(f"    {total_tokens} tokens → {len(chunks_tokens)} chunks "
          f"of ~{MAX_CHUNK_TOKENS} tokens each")

    chunk_vectors = []

    for i, chunk_tokens in enumerate(chunks_tokens):
        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        try:
            vec = embed_single_chunk(chunk_text)
            chunk_vectors.append(vec)

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"    OOM on chunk {i+1}/{len(chunks_tokens)} — clearing and retrying")
                torch.cuda.empty_cache()
                gc.collect()
                torch.cuda.empty_cache()

                try:
                    vec = embed_single_chunk(chunk_text)
                    chunk_vectors.append(vec)
                    print(f"    Chunk {i+1} retry succeeded")

                except RuntimeError:
                    print(f"    Chunk {i+1} failed on retry — skipping")
                    continue
            else:
                raise e

    if len(chunk_vectors) == 0:
        return None

    return np.mean(chunk_vectors, axis=0)


print("Embedding functions ready")

Embedding functions ready


In [ ]:
from tqdm import tqdm

SAVE_EVERY = 10
OUTPUT_NPY = "/content/drive/MyDrive/script_embeddings.npy"
OUTPUT_CSV = "/content/drive/MyDrive/movies_with_embeddings.csv"

# --- Resume from checkpoint if one exists ---
if os.path.exists(OUTPUT_NPY) and os.path.exists(OUTPUT_CSV):
    embeddings = list(np.load(OUTPUT_NPY))
    df_done = pd.read_csv(OUTPUT_CSV)
    done_titles = set(df_done["title"].tolist())
    valid_rows = df_done.to_dict("records")
    print(f"Resuming from checkpoint — {len(embeddings)} scripts already done")
    print(f"Remaining: {len(df) - len(done_titles)} scripts")
else:
    embeddings = []
    valid_rows = []
    done_titles = set()
    print(f"Starting fresh — {len(df)} scripts to process")

# --- Main loop ---
skipped_missing = 0
skipped_oom = 0
skipped_short = 0

for idx, row in tqdm(df.iterrows(), total=len(df)):
    movie = row["title"]

    # Skip already processed
    if movie in done_titles:
        continue

    path = row["script_file"]

    # Read script
    script_text = read_script(path)

    if script_text is None:
        skipped_missing += 1
        continue

    if len(script_text.split()) < 50:
        print(f"  SKIPPING {movie} — too short ({len(script_text.split())} words)")
        skipped_short += 1
        continue

    # Embed
    print(f"Processing: {movie}")
    vec = None

    try:
        vec = embed_script(script_text)

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"  OOM on {movie} — performing full reset")
            torch.cuda.empty_cache()
            gc.collect()
            torch.cuda.empty_cache()
            skipped_oom += 1
        else:
            print(f"  Unexpected error on {movie}: {e}")

    finally:
        # Always clear after every script regardless of outcome
        del script_text
        torch.cuda.empty_cache()
        gc.collect()

    # Store result if successful
    if vec is not None:
        if np.isnan(vec).any():
            print(f"  WARNING: NaN vector for {movie} — discarding")
        else:
            embeddings.append(vec)
            valid_rows.append(row.to_dict())
            done_titles.add(movie)
            print(f"  Done — shape: {vec.shape}, "
                  f"mean: {vec.mean():.4f}, std: {vec.std():.4f}")

    # Incremental checkpoint
    if len(embeddings) > 0 and len(embeddings) % SAVE_EVERY == 0:
        np.save(OUTPUT_NPY, np.stack(embeddings))
        pd.DataFrame(valid_rows).to_csv(OUTPUT_CSV, index=False)
        print(f"\n--- Checkpoint saved: {len(embeddings)} embeddings ---\n")

# Final save
if len(embeddings) > 0:
    np.save(OUTPUT_NPY, np.stack(embeddings))
    pd.DataFrame(valid_rows).to_csv(OUTPUT_CSV, index=False)
    print(f"Final save complete — {len(embeddings)} embeddings written")
else:
    print("WARNING: No embeddings to save")

print(f"""
Done.
  Successful:             {len(embeddings)}
  Skipped (missing file): {skipped_missing}
  Skipped (OOM):          {skipped_oom}
  Skipped (too short):    {skipped_short}
  Total in dataframe:     {len(df)}
""")

Starting fresh — 8592 scripts to process


  0%|          | 0/8592 [00:00<?, ?it/s]

Processing: The Place Beyond the Pines
    42377 tokens → 6 chunks of ~8000 tokens each


  0%|          | 1/8592 [00:12<30:08:57, 12.63s/it]

  Done — shape: (2560,), mean: -0.0002, std: 0.0179
Processing: The Man from U.N.C.L.E.
    36626 tokens → 5 chunks of ~8000 tokens each


...

  Done — shape: (2560,), mean: -0.0000, std: 0.0178
Processing: X-Men: Apocalypse
    46383 tokens → 6 chunks of ~8000 tokens each


  1%|          | 48/8592 [06:05<29:58:26, 12.63s/it]

  Done — shape: (2560,), mean: -0.0001, std: 0.0187

--- Checkpoint saved: 30 embeddings ---

Processing: Christy
    34971 tokens → 5 chunks of ~8000 tokens each


  1%|          | 48/8592 [06:17<18:39:21,  7.86s/it]


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')